In [1]:
!pip install -q numpy scikit-learn sentence-transformers

In [2]:
%%writefile search_module.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Finder:
    def __init__(self, data, strategy="tfidf"):
        if not data or any(not item["text"].strip() for item in data):
            raise ValueError("Data cannot be empty")
        self.data, self.strategy = data, strategy
        corpus = [item["text"] for item in data]
        if strategy == "tfidf":
            self.vec = TfidfVectorizer()
            self.mat = self.vec.fit_transform(corpus)
        elif strategy == "neural":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.verify_len(corpus)
            self.mat = self.model.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Choose tfidf or neural")
    def verify_len(self, texts):
        for t in texts:
            toks = len(self.model.tokenizer(t, truncation=False)["input_ids"])
            if toks > self.model.max_seq_length:
                raise ValueError("Token limit exceeded")
    def retrieve(self, q, k=3, criteria=None):
        if not q.strip() or k <= 0: return []
        if self.strategy == "tfidf":
            qv = self.vec.transform([q])
            if qv.nnz == 0: return []
            scores = (qv @ self.mat.T).toarray()[0]
        else:
            self.verify_len([q])
            qv = self.model.encode([q], normalize_embeddings=True)[0]
            scores = self.mat @ qv
        valid = [idx for idx, row in enumerate(self.data)
                   if all(row.get(key) == val for key, val in (criteria or {}).items())]
        ranked = sorted(valid, key=lambda idx: (-float(scores[idx]), idx))
        return [(self.data[i], float(scores[i])) for i in ranked[:k]]
def display_hits(results):
    if not results:
        print("No matching docs.")
    for r, sc in results:
        print(r["id"], f"confidence={sc:.3f}")
        print(r["text"])

Writing search_module.py


In [3]:
%%writefile text_splitter.py
import re
def fetch_words(doc):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", doc)]
def block_window(doc, cap=40, overlap=10, shift=0):
    if cap <= 0 or not 0 <= overlap < cap:
        raise ValueError("Require cap > 0 and 0 <= overlap < cap")
    spans = fetch_words(doc)
    res = []
    for i in range(0, len(spans), cap - overlap):
        j = min(i + cap, len(spans))
        res.append((shift + spans[i][0], shift + spans[j - 1][1]))
        if j == len(spans): break
    return res
def smart_split(doc, cap=40):
    if cap <= 0: raise ValueError("Limit must be positive")
    breaks = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", doc)] + [len(doc)]
    chunks, buf, cnt = [], None, 0
    for a, b in zip(breaks, breaks[1:]):
        seg = doc[a:b]
        num = len(seg.split())
        if not num: continue
        if num > cap:
            if buf: chunks.append(buf)
            chunks.extend(block_window(seg, cap, 0, a))
            buf, cnt = None, 0
        elif buf and cnt + num <= cap:
            buf, cnt = (buf[0], b), cnt + num
        else:
            if buf: chunks.append(buf)
            buf, cnt = (a, b), num
    if buf: chunks.append(buf)
    return chunks
def create_segments(corpus, mode):
    output = []
    for d_id, content in corpus.items():
        bounds = smart_split(content) if mode == "smart" else block_window(content, 40, int(mode))
        for idx, (x, y) in enumerate(bounds):
            output.append(dict(id=f"{d_id}_{idx}", ref=d_id, start=x, end=y, text=content[x:y]))
    return output
if __name__ == "__main__":
    assert block_window("") == []
    assert block_window("short text") == [(0, 10)]
    try: block_window("a b", 2, 2)
    except ValueError: pass
    long_txt = " ".join(["token"] * 95) + "."
    assert all(len(long_txt[a:b].split()) <= 40 for a, b in smart_split(long_txt))
    ds = {"test_doc": "Reboot the hub. Keep passwords secret."}
    for s in ["0", "10", "smart"]:
        for c in create_segments(ds, s):
            assert c["text"] == ds[c["ref"]][c["start"]:c["end"]]

Writing text_splitter.py


In [4]:
%%writefile doc_analysis.py
from text_splitter import create_segments, block_window, smart_split
IoT_DOCS = {
    "wifi": (
        "Network Connectivity Guide. Smart devices require a stable internet "
        "connection to function correctly. Most of our smart home hubs only "
        "support 2.4GHz Wi-Fi networks due to its better range through walls. "
        "If your router combines both 2.4GHz and 5GHz under one name, you might "
        "need to temporarily disable the 5GHz band during initial setup. "
        "To connect, open the mobile app, tap the add device icon, and enter "
        "your network password. Ensure your phone is connected to the same "
        "network before starting. If the device flashes a red LED, it means "
        "the password was incorrect or the signal is too weak. Moving the "
        "device closer to the router during setup often solves range issues. "
        "After successful pairing, the LED will turn solid green for five "
        "seconds and then turn off to save power. MAC filtering on your router "
        "must be turned off or updated with the device's MAC address, which "
        "is printed on the bottom sticker."
    ),
    "reset": (
        "Device Factory Reset Instructions. If your smart device becomes "
        "unresponsive or you wish to transfer ownership, a factory reset is "
        "necessary. Before proceeding, note that resetting will erase all "
        "custom schedules, user profiles, and network settings permanently. "
        "To initiate the reset, locate the small pinhole button on the back "
        "of the device. Using a paperclip or a similar tool, press and hold "
        "the button for exactly fifteen seconds while the device is plugged in. "
        "The status lights will rapidly flash amber to indicate the process "
        "has begun. Release the button and wait for two minutes while the "
        "internal memory is cleared and the firmware reboots. Once you hear "
        "a chime and see a slow pulsing blue light, the reset is complete. "
        "Do not unplug the power adapter during this process, as it may "
        "corrupt the operating system. After the reset, you must remove the "
        "device from your mobile app manually before it can be registered again."
    ),
    "camera": (
        "Security Camera Storage Options. Our indoor and outdoor security "
        "cameras offer two primary methods for saving recorded video footage. "
        "The standard method relies on an onboard MicroSD card slot. The cameras "
        "support cards up to two hundred and fifty-six gigabytes, formatted in "
        "FAT32. Local storage automatically overwrites the oldest clips when "
        "full. Alternatively, users can subscribe to our Secure Cloud service. "
        "The basic cloud tier retains motion events for seven days, while the "
        "premium tier offers thirty days of continuous twenty-four-hour recording. "
        "Cloud storage requires a minimum upload speed of two megabits per "
        "second to function reliably. If the internet connection drops, the "
        "camera will fall back to local storage if a card is present. Footage "
        "saved to the cloud is encrypted end-to-end and can be downloaded "
        "directly to your smartphone through the companion app. Video quality "
        "settings can be adjusted to save space, but higher resolutions will "
        "consume storage bandwidth much faster."
    )
}
KEY_FACTS = {
    "wifi": ("Most of our smart home hubs only support 2.4GHz Wi-Fi networks",
             "Moving the device closer to the router during setup often solves"),
    "reset": ("resetting will erase all custom schedules, user profiles, and network",
              "press and hold the button for exactly fifteen seconds while the device is plugged in"),
    "camera": ("The basic cloud tier retains motion events for seven days",
               "Cloud storage requires a minimum upload speed of two megabits"),
}
if __name__ == "__main__":
    w_counts = {k: len(v.split()) for k, v in IoT_DOCS.items()}
    for k, v in IoT_DOCS.items():
        max_len = max(len(s.split()) for s in v.replace("?", ".").split("."))
        print(k, "word count:", w_counts[k], "| max sentence:", max_len)
        assert k in KEY_FACTS and all(p in v for p in KEY_FACTS[k])
    total_w = sum(w_counts.values())
    print()
    for strategy, desc in [("0", "win 40, no overlap"), ("10", "win 40, overlap 10"), ("smart", "smart 40")]:
        frags = create_segments(IoT_DOCS, strategy)
        sizes = [len(f["text"].split()) for f in frags]
        dup = sum(sizes) - total_w
        surv = sum(1 for d, (f1, f2) in KEY_FACTS.items() if any(c["ref"] == d and f1 in c["text"] and f2 in c["text"] for c in frags))
        print(f"{desc}: frags={len(frags)}, avg words={sum(sizes)/len(sizes):.1f}, duplicate words={dup}, pairs intact={surv}/{len(KEY_FACTS)}")
        for c in frags:
            assert c["text"] == IoT_DOCS[c["ref"]][c["start"]:c["end"]]

Writing doc_analysis.py


In [5]:
!ls

doc_analysis.py  sample_data  search_module.py	text_splitter.py


In [6]:
%%writefile build_kb.py
import json
from pathlib import Path
from doc_analysis import IoT_DOCS

data_store = dict(IoT_DOCS)
data_store["thermostat"] = (
    "Thermostat Climate Control Protocol. This document outlines how the smart "
    "thermostat regulates ambient temperature and interacts with HVAC systems. "
    "The device uses dual sensors to measure both temperature and humidity, "
    "adjusting output to optimize comfort. Users can configure a weekly "
    "schedule with up to four distinct periods per day: morning, away, "
    "evening, and sleep. The Eco mode override can be activated automatically "
    "using geofencing when the house is empty. Unlike traditional thermostats, "
    "this unit requires a C-wire for continuous power and will not function "
    "on batteries alone. If the display shows an E73 error, it indicates "
    "a power interruption from the furnace control board. Firmware updates "
    "are downloaded overnight between 2 AM and 4 AM to minimize disruptions. "
    "Compatibility extends to most 24V conventional and heat pump systems, "
    "but proprietary communicating systems will require a special adapter "
    "module installed by a certified technician."
)
data_store["lock"] = (
    "Smart Lock Mechanical and Digital Guide. The biometric smart lock "
    "provides keyless entry while retaining a physical override for safety. "
    "The motorized deadbolt is powered by four AA alkaline batteries, which "
    "typically last for eight to ten months under normal household usage. "
    "When battery life drops below twenty percent, the keypad will flash red "
    "and send a push notification. Lithium batteries are strictly prohibited "
    "because they may deliver a voltage spike that damages the mainboard. "
    "If the batteries die completely, a 9V battery can be held against the "
    "two terminals at the bottom of the exterior casing to provide temporary "
    "power. Up to fifty unique fingerprint profiles and one hundred numeric "
    "PIN codes can be stored simultaneously. The auto-lock feature secures "
    "the door automatically three minutes after it is closed, though this "
    "timer can be adjusted in the settings menu. A hidden mechanical keyhole "
    "is located behind the magnetic faceplate for emergencies."
)

evidences = {
    "wifi": "Most of our smart home hubs only support 2.4GHz Wi-Fi networks",
    "reset": "press and hold the button for exactly fifteen seconds",
    "camera": "The basic cloud tier retains motion events for seven days",
    "thermostat": "this unit requires a C-wire for continuous power and will not function on batteries alone.",
    "lock": "Lithium batteries are strictly prohibited because they may deliver a voltage spike",
}

test_set = [
    ("What Wi-Fi frequency is required?", "wifi", "val"),
    ("How long should I hold the reset button?", "reset", "val"),
    ("How long are clips saved in the basic plan?", "camera", "val"),
    ("Can the thermostat run on batteries?", "thermostat", "val"),
    ("Why can't I use lithium batteries in the lock?", "lock", "test"),
    ("How do I fix an E73 error?", "thermostat", "test"),
    ("What happens if my camera loses internet?", "camera", "test"),
    ("How do I connect to a hidden network?", "wifi", "test"),
    ("What is the weather like tomorrow?", None, "test"),
    ("Who is the CEO of the company?", None, "test")
]

queries_list = [dict(q=q, doc=doc, evidence=evidences[doc] if doc else None, split=split) for q, doc, split in test_set]

for k, v in evidences.items():
    assert v in data_store[k], f"Missing evidence in {k}"

for f_name, content in [("kb_data.json", data_store), ("eval_queries.json", queries_list)]:
    if Path(f_name).exists(): raise FileExistsError(f"{f_name} exists")
    Path(f_name).write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")
print("kb_data.json and eval_queries.json generated.")

Writing build_kb.py


In [7]:
!python build_kb.py

kb_data.json and eval_queries.json generated.


In [8]:
%%writefile eval_system.py
import csv, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from text_splitter import create_segments

def score_results(scores_matrix, fragments, questions):
    h, mrr = [], []
    for row, q in zip(scores_matrix, questions):
        if not q["doc"]: continue
        top = np.argsort(-row, kind="stable")[:3]
        hits = [r for r, i in enumerate(top, 1) if fragments[i]["ref"] == q["doc"] and q["evidence"] in fragments[i]["text"]]
        h.append(bool(hits))
        mrr.append(1/hits[0] if hits else 0)
    return (float(np.mean(h)), float(np.mean(mrr))) if h else (None, None)

def execute_eval():
    with open("kb_data.json", encoding="utf-8") as f: data = json.load(f)
    with open("eval_queries.json", encoding="utf-8") as f: q_data = json.load(f)

    t0 = time.perf_counter()
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    print("Encoder init time:", time.perf_counter() - t0)

    q_strings = [q["q"] for q in q_data]
    history = []

    for strat in ["0", "10", "smart"]:
        frags = create_segments(data, strat)
        f_texts = [f["text"] for f in frags]
        tok_lens = [len(encoder.tokenizer(txt, truncation=False)["input_ids"]) for txt in f_texts + q_strings]
        print(f"Strat: {strat} | Frags: {len(frags)} | Max toks: {max(tok_lens)}")

        for algo in ["tfidf", "neural"]:
            t1 = time.perf_counter()
            if algo == "tfidf":
                vec = TfidfVectorizer()
                corpus_mat = vec.fit_transform(f_texts)
                q_mat = vec.transform(q_strings)
                sims = (q_mat @ corpus_mat.T).toarray()
            else:
                corpus_mat = encoder.encode(f_texts, normalize_embeddings=True)
                q_mat = encoder.encode(q_strings, normalize_embeddings=True)
                sims = q_mat @ corpus_mat.T
            calc_time = time.perf_counter() - t1

            for split in ["val", "test"]:
                idxs = [i for i, x in enumerate(q_data) if x["split"] == split]
                res = score_results(sims[idxs], frags, [q_data[i] for i in idxs])
                print(f"{strat} {algo} {split} Performance:", res)

            for row, q in zip(sims, q_data):
                for r_idx, f_idx in enumerate(np.argsort(-row, kind="stable")[:3], 1):
                    fg = frags[f_idx]
                    history.append({
                        "split_strat": strat, "model": algo, "query": q["q"], "partition": q["split"],
                        "rank": r_idx, "conf": float(row[f_idx]), "frag_id": fg["id"],
                        "doc_ref": fg["ref"], "char_start": fg["start"], "char_end": fg["end"],
                        "snippet": fg["text"]
                    })

    with open("search_metrics.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(history[0]))
        w.writeheader()
        w.writerows(history)
    print("Logged results to search_metrics.csv")

if __name__ == "__main__":
    execute_eval()

Writing eval_system.py


In [9]:
!python eval_system.py

modules.json: 100% 349/349 [00:00<00:00, 1.20MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 400kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 16.0MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 223kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.46MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 113MB/s, 7.05MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 61.4MB/s, 8.18MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 65.7MB/s, 8.82MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 5087.84it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.34MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 12.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 34.1MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 438kB/s]
config.json: 100% 190/190 [00:00<00:00, 683kB/s]
Encoder init time: 4.069751874999952
Strat: 0 | Frags: 20 | Max toks: 59
0 tfidf val Performance: (0.75,

In [10]:
%%writefile interactive_bot.py
import json
from text_splitter import create_segments
from search_module import Finder

if __name__ == "__main__":
    with open("kb_data.json", encoding="utf-8") as f:
        docs = json.load(f)
    engine = Finder(create_segments(docs, "smart"), "neural")
    print("IoT Support Bot Initialized. Type 'quit' to exit.")

    while True:
        try: user_in = input("User Query: ")
        except (EOFError, KeyboardInterrupt): break
        if user_in.strip().lower() == "quit": break

        results = engine.retrieve(user_in)
        if not results:
            print("No relevant documentation found. Escalate to tier 2 support.")
            continue

        print("Top suggestions (verify before replying):")
        for i, (res, conf) in enumerate(results, 1):
            print(f"[{i}] {res['id']} (Match: {conf:.2f}) [Chars {res['start']}-{res['end']}]")

        sel = input("Select a number to expand (1-3) or press Enter to skip: ")
        if sel.isdigit() and 1 <= int(sel) <= len(results):
            res, conf = results[int(sel) - 1]
            print(f"\n--- {res['ref'].upper()} DOC EXCERPT ---")
            print(res['text'])
            print("------------------------\n")
        else:
            print("Selection skipped.\n")

Writing interactive_bot.py


In [11]:
%run interactive_bot.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

IoT Support Bot Initialized. Type 'quit' to exit.
User Query: exit
Top suggestions (verify before replying):
[1] lock_3 (Match: 0.20) [Chars 490-639]
[2] reset_2 (Match: 0.18) [Chars 468-653]
[3] lock_5 (Match: 0.18) [Chars 886-971]
Select a number to expand (1-3) or press Enter to skip: 2

--- RESET DOC EXCERPT ---
The status lights will rapidly flash amber to indicate the process has begun. Release the button and wait for two minutes while the internal memory is cleared and the firmware reboots. 
------------------------



In [12]:
from google.colab import files
files.download("kb_data.json")
files.download("eval_queries.json")
files.download("search_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>